In [ ]:
# | default_exp visualization

# Visualization Tool

> Defining tools for model visualization

In [ ]:
# | export
import matplotlib.pyplot as plt
import numpy as np
import torch
import seaborn as sns
from matrepr import mprint
from torch import nn
from matplotlib.animation import FuncAnimation
import warnings
from learningdynamics.utils import getActivation

In [ ]:
# | export
def plot_decision_boundary_movie(
    model: nn.Module,  # PyTorch model.
    state_dicts: list[nn.Module.state_dict],  # List of state dicts to build decision boundary
    X: torch.Tensor,  # Input data
    y: torch.Tensor,  # Label vector
    ax: plt.Axes,  # Axes object to plot on
    labels: list[int] = [0, 1],  # Labels
):
    # Set the overall theme without a grid
    sns.set_context("paper")
    sns.set_style("white")
    ax.set_aspect("equal")

    # Get Seaborn's tab colors
    tab10 = sns.color_palette("tab10")
    orange = tab10[1]
    blue = tab10[0]

    # Initialization
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
    x_in = np.c_[xx.ravel(), yy.ravel()]
    x_in = torch.tensor(x_in, dtype=torch.float32).to(
        next(model.parameters()).device
    )

    # Figure and plot
    sns.scatterplot(
        x=X[y == labels[0], 0].cpu().numpy(),
        y=X[y == labels[0], 1].cpu().numpy(),
        ax=ax,
        color=orange,
        marker="o",
        s=50,
    )
    sns.scatterplot(
        x=X[y == labels[1], 0].cpu().numpy(),
        y=X[y == labels[1], 1].cpu().numpy(),
        ax=ax,
        color=blue,
        marker="o",
        s=50,
    )

    # Initialize `contour` outside of the update function
    contour = ax.contourf(
        xx,
        yy,
        np.zeros_like(xx),
        colors=[orange, blue],
        alpha=0.3,
        levels=np.linspace(labels[0], labels[1], 3),
    )

    # Precompute predictions for all state_dicts
    y_preds = []
    for model_state_dict in state_dicts:
        model.load_state_dict(model_state_dict)
        model.eval()
        out = model(x_in)
        out = torch.argmax(out, dim=1)
        with torch.no_grad():
            y_preds.append(out.cpu().numpy().reshape(xx.shape))

    # Update function with precomputed predictions
    def update(frame):
        nonlocal contour  # Use the outer scope's contour variable

        y_pred = y_preds[frame]

        for coll in contour.collections:
            coll.remove()
        contour = ax.contourf(
            xx,
            yy,
            y_pred,
            colors=[orange, blue],
            alpha=0.3,
            levels=np.linspace(labels[0], labels[1], 3),
        )
        return contour.collections

    # Animation
    ani = FuncAnimation(ax.figure, update, frames=len(state_dicts), blit=True, repeat=False)
    return ani


In [ ]:
# | export
def koopman_inference(
    x_in,
    nn_model,
    dmd_model,
    layer_start,
    num_iterations,
    num_delays,
    num_usable_states
):
        
    hooks = []
    activation_dict = {}
    nn_model.eval()

    # Attach hooks to scaled model
    for i in range(layer_start, layer_start + (num_iterations-1) * 2, 2):
        hooks.append(
            nn_model.features[i].register_forward_hook(
                getActivation(f"layer_{i}", activation_dict)
            )
        )
    
    final_koop_out = []
    _ = nn_model(x_in)

    # Grab activation
    out = torch.stack(list(activation_dict.values()))  # shape = i,t,s

    koop_list = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for traj in range(out.shape[1]):
            traj_in = out[:num_delays+1, traj, :].numpy()

            Xkoop = dmd_model.simulate(
                traj_in,
                n_steps=(num_iterations) - 1 - num_delays
            )

            koop_list.append(torch.from_numpy(Xkoop)[-1:, :])

        koop_out = torch.cat(koop_list)[:, :num_usable_states]
        koop_out = torch.nn.functional.relu(koop_out)
    
    # Feed through the rest
    for i in range(
        layer_start + (num_iterations-1) * 2 + 1, len(nn_model.features)
    ):
        koop_out = nn_model.features[i](koop_out)

    final_koop_out.append(koop_out)
    final_koop_out = torch.cat(final_koop_out)

    return final_koop_out


In [ ]:
# | export
def plot_decision_boundary_koopman(
    model: nn.Module,  # PyTorch model.
    state_dicts: list[nn.Module.state_dict],  # List of state dicts to build decision boundary
    X: torch.Tensor,  # Input data
    y: torch.Tensor,  # Label vector
    dmd_model,
    layer_start,
    num_iterations,
    num_delays,
    num_usable_states,
    ax: plt.Axes,  # Axes object to plot on
    labels: list[int] = [0, 1],  # Labels
):
  
    # Set the overall theme without a grid
    sns.set_context("paper")
    sns.set_style("white")
    ax.set_aspect("equal")

    # Get Seaborn's tab colors
    tab10 = sns.color_palette("tab10")
    orange = tab10[1]
    blue = tab10[0]

    # Initialization
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
    x_in = np.c_[xx.ravel(), yy.ravel()]
    x_in = torch.tensor(x_in, dtype=torch.float32).to(
        next(model.parameters()).device
    )

    # Figure and plot
    sns.scatterplot(
        x=X[y == labels[0], 0].cpu().numpy(),
        y=X[y == labels[0], 1].cpu().numpy(),
        ax=ax,
        color=orange,
        marker="o",
        s=50,
    )
    sns.scatterplot(
        x=X[y == labels[1], 0].cpu().numpy(),
        y=X[y == labels[1], 1].cpu().numpy(),
        ax=ax,
        color=blue,
        marker="o",
        s=50,
    )

    # Initialize `contour` outside of the update function
    contour = ax.contourf(
        xx,
        yy,
        np.zeros_like(xx),
        colors=[orange, blue],
        alpha=0.3,
        levels=np.linspace(labels[0], labels[1], 2),
    )

    # Precompute predictions for all state_dicts
    y_preds = []
    for model_state_dict in state_dicts:
        model.load_state_dict(model_state_dict)
        model.eval()
        out = koopman_inference(
            x_in=x_in,
            nn_model=model,
            dmd_model=dmd_model,
            layer_start=layer_start,
            num_iterations=num_iterations,
            num_delays=num_delays,
            num_usable_states=num_usable_states,
        )
        out = torch.argmax(out, dim=1)
        
        with torch.no_grad():
            y_preds.append(out.cpu().numpy().reshape(xx.shape))
    
        
    # Update function with precomputed predictions
    def update(frame):
        nonlocal contour  # Use the outer scope's contour variable

        y_pred = y_preds[frame]

        for coll in contour.collections:
            coll.remove()
        contour = ax.contourf(
            xx,
            yy,
            y_pred,
            colors=[orange, blue],
            alpha=0.3,
            levels=np.linspace(labels[0], labels[1], 3),
        )
        return contour.collections

    # Animation
    ani = FuncAnimation(ax.figure, update, frames=len(state_dicts), blit=True, repeat=False)
    return ani